    # Notebook 3 — Les modèles multifactoriels

    **Le Quotient Financier · Travaux pratiques d'histoire de la gestion de portefeuille**

    Le modèle de Sharpe oublie ce que deux banques ou deux pétrolières partagent au-delà du marché.
    Ses successeurs ajoutent des facteurs : indices sectoriels (Cohen et Pogue, 1967), modèles de risque
    commerciaux (Barra, années 1970), facteurs statistiques, facteurs de style (Fama et French, 1993).
    Ce notebook construit ces modèles et pose la question qui compte aujourd'hui : **lequel estime le mieux le risque ?**

    **Au programme**

    1. Construire des facteurs sectoriels et un modèle à 6 facteurs.
    2. Mesurer la correction du biais sectoriel.
    3. Inverser la matrice avec la formule de Woodbury.
    4. Extraire des facteurs statistiques par analyse en composantes principales.
    5. Combiner les estimateurs : le rétrécissement de Ledoit et Wolf (2003).
    6. Une expérience contrôlée : 100 titres simulés, vérité connue.
    7. L'épreuve hors échantillon sur les vraies données.

    **Prérequis** : les notebooks 1 et 2.

> *Ce notebook est un support pédagogique du Quotient Financier. Il illustre des méthodes historiques
> et académiques ; il ne constitue en aucun cas un conseil en investissement ni une recommandation
> d'achat ou de vente de titres. Les titres utilisés servent uniquement d'exemples.*

## 0. Mise en place

In [ ]:
# Sur Google Colab ou une installation neuve, décommentez la ligne suivante :
# !pip install yfinance numpy pandas scipy matplotlib

import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # masque les avertissements de polices
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "mathtext.fontset": "dejavusans"})   # formules des axes (ex. 10^1) avec la police par défaut
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.set_printoptions(precision=4, suppress=True)

SEMAINES = 52              # nombre de semaines par an, pour annualiser
TAUX_SANS_RISQUE = 0.02    # taux sans risque annuel retenu (hypothèse simplificatrice)

In [ ]:
# Dix titres américains, cinq secteurs, deux titres par secteur
TITRES = {
    "AAPL": "Technologie", "MSFT": "Technologie",
    "JPM": "Banque",       "BAC": "Banque",
    "XOM": "Énergie",      "CVX": "Énergie",
    "JNJ": "Santé",        "PFE": "Santé",
    "KO": "Consommation",  "PG": "Consommation",
}
MARCHE = "SPY"   # ETF S&P 500, utilisé comme indice de marché
ETF_SECTEURS = {"XLK": "Technologie", "XLF": "Banque", "XLE": "Énergie",
                "XLV": "Santé", "XLP": "Consommation"}

SECTEURS = {**TITRES, **ETF_SECTEURS, MARCHE: "Marché"}
TOUS = list(TITRES) + [MARCHE] + list(ETF_SECTEURS)
DEBUT, FIN = "2015-01-01", "2024-12-31"
FICHIER = "donnees_prix.csv"

In [ ]:
def donnees_simulees(tickers, secteurs, debut, fin, graine=42):
    """Prix hebdomadaires simulés par un modèle à facteurs (marché + secteurs + bruit à queues épaisses).
    Utilisés uniquement si le téléchargement échoue, pour que le notebook reste exécutable."""
    rng = np.random.default_rng(graine)
    dates = pd.date_range(debut, fin, freq="W-FRI")
    T = len(dates)
    t5 = lambda: rng.standard_t(5, T) / np.sqrt(5 / 3)          # loi de Student, variance 1
    marche = 0.0025 + 0.022 * t5()
    facteurs = {s: 0.015 * t5() for s in set(secteurs.values())}
    rend = {}
    for t in tickers:
        s = secteurs.get(t, "Marché")
        if t == "SPY":
            r = marche + 0.001 * rng.standard_normal(T)
        elif t == "AGG":
            r = 0.0006 + 0.008 * rng.standard_normal(T) - 0.05 * marche
        elif t.startswith("XL"):
            r = rng.uniform(0.8, 1.2) * marche + facteurs[s] + 0.003 * rng.standard_normal(T)
        else:
            r = (0.0006 * rng.standard_normal() + rng.uniform(0.6, 1.4) * marche
                 + rng.uniform(0.8, 1.5) * facteurs[s] + 0.018 * t5())
        rend[t] = r
    return 100 * (1 + pd.DataFrame(rend, index=dates)).cumprod()


def charger_prix(tickers, secteurs, debut, fin, fichier):
    """Lit le fichier local s'il existe, sinon télécharge sur Yahoo Finance,
    sinon (pas d'accès internet) bascule sur des données simulées."""
    if os.path.exists(fichier):
        prix = pd.read_csv(fichier, index_col=0, parse_dates=True)
        if set(tickers) <= set(prix.columns):
            print(f"Données lues dans {fichier}.")
            return prix[tickers], "fichier local"
    try:
        import yfinance as yf
        prix = yf.download(tickers, start=debut, end=fin, auto_adjust=True, progress=False)["Close"]
        if prix is None or len(prix) == 0 or prix.dropna(how="all").empty:
            raise ValueError("aucune donnée reçue")
        prix = prix[tickers].dropna(how="all")
        manquants = [t for t in tickers if prix[t].isna().mean() > 0.05]
        if prix.empty or manquants:
            raise ValueError(f"séries incomplètes : {manquants}")
        prix.to_csv(fichier)
        print(f"Données téléchargées sur Yahoo Finance et enregistrées dans {fichier}.")
        return prix, "Yahoo Finance"
    except Exception as e:
        print("⚠️  Téléchargement impossible (" + str(e)[:90] + ").")
        print("⚠️  LE NOTEBOOK UTILISE DES DONNÉES SIMULÉES : les chiffres ne décrivent pas les vrais titres.")
        return donnees_simulees(tickers, secteurs, debut, fin), "simulées"


def rendements_hebdo(prix):
    """Rendements hebdomadaires simples, calculés sur les cours du vendredi."""
    return prix.resample("W-FRI").last().pct_change(fill_method=None).dropna(how="any")


def boites(ax, donnees, noms):
    """Boîtes à moustaches, une par colonne de `donnees` (compatible avec toutes les versions de matplotlib)."""
    ax.boxplot(np.asarray(donnees), showfliers=False)
    ax.set_xticks(range(1, len(noms) + 1))
    ax.set_xticklabels(list(noms), rotation=90)


def afficher_matrice(M, titre, vmin=-1, vmax=1, cmap="RdBu_r", fmt="{:.2f}"):
    """Affiche une matrice (DataFrame carré) sous forme de carte de chaleur annotée."""
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(M.values, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xticks(range(len(M))); ax.set_xticklabels(M.columns, rotation=90)
    ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index)
    for i in range(len(M)):
        for j in range(len(M)):
            ax.text(j, i, fmt.format(M.values[i, j]), ha="center", va="center", fontsize=7)
    ax.grid(False); ax.set_title(titre)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.show()

In [ ]:
# Outils d'optimisation (repris du notebook 1)
def tangent(Sigma_, mu_, rf=TAUX_SANS_RISQUE):
    """Portefeuille tangent, vente à découvert permise (formule fermée)."""
    x = np.linalg.solve(Sigma_, mu_ - rf)
    return x / x.sum()

def tangent_positif(Sigma_, mu_, rf=TAUX_SANS_RISQUE):
    """Portefeuille tangent à poids positifs (optimisation numérique)."""
    n = len(mu_)
    res = minimize(lambda w: -(w @ mu_ - rf) / np.sqrt(w @ Sigma_ @ w), np.ones(n) / n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   bounds=[(0, 1)] * n, method="SLSQP", options={"ftol": 1e-10, "maxiter": 500})
    return res.x

def variance_min(Sigma_):
    """Portefeuille de variance minimale, vente à découvert permise."""
    x = np.linalg.solve(Sigma_, np.ones(len(Sigma_)))
    return x / x.sum()

def variance_min_positive(Sigma_):
    """Portefeuille de variance minimale à poids positifs."""
    n = len(Sigma_)
    res = minimize(lambda w: w @ Sigma_ @ w, np.ones(n) / n, jac=lambda w: 2 * Sigma_ @ w,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   bounds=[(0, 1)] * n, method="SLSQP", options={"ftol": 1e-12, "maxiter": 500})
    return res.x

def performance(r, rf=TAUX_SANS_RISQUE):
    """Indicateurs annualisés d'une série de rendements hebdomadaires."""
    valeur = (1 + r).cumprod()
    return pd.Series({"Rendement annuel": r.mean() * SEMAINES,
                      "Volatilité": r.std() * np.sqrt(SEMAINES),
                      "Ratio de Sharpe": (r.mean() * SEMAINES - rf) / (r.std() * np.sqrt(SEMAINES)),
                      "Perte maximale": (valeur / valeur.cummax() - 1).min()})

In [ ]:
prix, SOURCE = charger_prix(TOUS, SECTEURS, DEBUT, FIN, FICHIER)
rendements = rendements_hebdo(prix)
rend = rendements[list(TITRES)].loc["2020":"2024"]
rm = rendements[MARCHE].loc["2020":"2024"]
retf = rendements[list(ETF_SECTEURS)].loc["2020":"2024"]
N, T = rend.shape[1], rend.shape[0]
Sigma = rend.cov() * SEMAINES
print(f"Source : {SOURCE} — {N} titres, {T} semaines (2020-2024)")

## 1. Construire les facteurs

Les ETF sectoriels sont eux-mêmes très corrélés au marché. Si on les utilisait tels quels à côté du marché,
les facteurs se chevaucheraient. On construit donc des **facteurs sectoriels purs** : la part du rendement
de chaque ETF que le marché n'explique pas, c'est-à-dire le résidu de sa régression sur `SPY`.

Le modèle s'écrit, avec $K = 6$ facteurs (le marché et cinq secteurs purs) :

$$R = a + B\, f + e, \qquad \Sigma = B\, \Sigma_F\, B^\top + D$$

où $B$ est la matrice $N \times K$ des expositions, $\Sigma_F$ la covariance des facteurs et $D$ la diagonale
des variances résiduelles.

### Exercice 1.1
Construisez la matrice des facteurs `F` (une colonne pour le marché, une par secteur pur).

In [ ]:
def residus_sur(y, x):
    """Résidus de la régression de chaque colonne de y sur une constante et x."""
    X = np.column_stack([np.ones(len(x)), x])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    return y - X @ coef

# À COMPLÉTER : facteurs sectoriels purs (résidus des ETF sur le marché), puis F = [marché, secteurs purs]
secteurs_purs = ...
F = ...
F.corr().style.format("{:.2f}")

### Exercice 1.2 — Les expositions
Régressez chaque titre sur les 6 facteurs, puis construisez `Sigma_multi` (annualisée).

In [ ]:
X = np.column_stack([np.ones(T), F.values])
# À COMPLÉTER : coefficients des régressions, matrice B (N x K), variances résiduelles, Sigma_F, Sigma_multi
B = ...
d_multi = ...
Sigma_F = ...
Sigma_multi = ...

## 2. Le biais sectoriel est-il corrigé ?

On compare trois matrices : l'empirique (aucune structure), celle de Sharpe (1 facteur) et la multifactorielle (6 facteurs).

In [ ]:
rm_ann = rm.var() * SEMAINES
Xm = np.column_stack([np.ones(T), rm.values])
cm, *_ = np.linalg.lstsq(Xm, rend.values, rcond=None)
b1 = cm[1]
Sigma_sharpe = rm_ann * np.outer(b1, b1) + np.diag((rend.values - Xm @ cm).var(axis=0, ddof=1) * SEMAINES)

def en_correlation(S_):
    s_ = np.sqrt(np.diag(S_))
    return pd.DataFrame(S_ / np.outer(s_, s_), index=TITRES, columns=TITRES)

corr = {"Empirique": rend.corr(), "Sharpe (1 facteur)": en_correlation(Sigma_sharpe),
        "Multifactoriel (6 facteurs)": en_correlation(Sigma_multi)}
paires = [("AAPL", "MSFT"), ("JPM", "BAC"), ("XOM", "CVX"), ("JNJ", "PFE"), ("KO", "PG"), ("AAPL", "XOM")]
tableau = pd.DataFrame({k: [v.loc[p] for p in paires] for k, v in corr.items()},
                       index=[f"{a} / {b}" for a, b in paires])
hors_diag = ~np.eye(N, dtype=bool)
for k in ("Sharpe (1 facteur)", "Multifactoriel (6 facteurs)"):
    ecart = np.abs(corr[k].values - corr["Empirique"].values)[hors_diag].mean()
    print(f"{k:<28} écart moyen de corrélation avec l'empirique : {ecart:.3f}")
tableau

### Question 2.1 — Combien de paramètres ?
Comptez les paramètres à estimer pour la matrice de covariance avec $N$ titres et $K$ facteurs :
$N \times K$ expositions, $K(K+1)/2$ covariances de facteurs, $N$ variances résiduelles.

In [ ]:
def n_params(n, k=None):
    if k is None:
        return n * (n + 1) // 2           # matrice empirique complète
    return n * k + k * (k + 1) // 2 + n

pd.DataFrame({"Empirique": [n_params(n) for n in (10, 100, 1000, 5000)],
              "Sharpe (K = 1)": [n_params(n, 1) for n in (10, 100, 1000, 5000)],
              "6 facteurs": [n_params(n, 6) for n in (10, 100, 1000, 5000)]},
             index=pd.Index([10, 100, 1000, 5000], name="Titres")).style.format("{:,.0f}")

## 3. Inverser avec Woodbury

La formule de Woodbury généralise Sherman-Morrison :

$$\Sigma^{-1} = D^{-1} - D^{-1} B \left(\Sigma_F^{-1} + B^\top D^{-1} B\right)^{-1} B^\top D^{-1}$$

La seule vraie inversion porte sur une matrice $K \times K$.

### Exercice 3.1
Écrivez `woodbury_resoudre(B, Sigma_F, d, v)` qui calcule $\Sigma^{-1}v$ sans former $\Sigma$,
vérifiez-la sur `Sigma_multi`, puis comparez les temps de calcul.

In [ ]:
def woodbury_resoudre(B_, Sigma_F_, d_, v):
    # À COMPLÉTER
    ...

v = np.ones(N)
print("Écart avec l'inversion directe :",
      np.abs(woodbury_resoudre(B, Sigma_F, d_multi, v) - np.linalg.solve(Sigma_multi, v)).max())

In [ ]:
def chrono(f, repetitions=3):
    t0 = time.perf_counter()
    for _ in range(repetitions):
        f()
    return (time.perf_counter() - t0) / repetitions

rng = np.random.default_rng(5)
lignes = []
for n in (100, 1000, 3000, 100_000):
    B_n = rng.normal(0, 0.5, (n, 6)); d_n = rng.uniform(0.02, 0.09, n); v_n = np.ones(n)
    t_w = chrono(lambda: woodbury_resoudre(B_n, Sigma_F, d_n, v_n))
    t_p = chrono(lambda: np.linalg.solve(B_n @ Sigma_F @ B_n.T + np.diag(d_n), v_n)) if n <= 3000 else np.nan
    lignes.append({"Titres": n, "Matrice pleine (ms)": 1000 * t_p, "Woodbury (ms)": 1000 * t_w})
pd.DataFrame(lignes).set_index("Titres").style.format("{:.2f}", na_rep="(mémoire insuffisante)")

## 4. Des facteurs statistiques : l'analyse en composantes principales

Au lieu de choisir les facteurs à la main, on peut les extraire des données. La matrice de covariance se
décompose en valeurs propres $\lambda_k$ et vecteurs propres $v_k$ :

$$\Sigma = \sum_{k=1}^{N} \lambda_k\, v_k v_k^\top$$

Chaque vecteur propre est un portefeuille, et $\lambda_k$ sa variance. On garde les $K$ premiers comme facteurs,
et on traite le reste comme du risque propre.

### Exercice 4.1
Calculez valeurs et vecteurs propres de `Sigma`, la part de variance expliquée, et tracez les poids des trois premiers vecteurs.

In [ ]:
# À COMPLÉTER : np.linalg.eigh, tri décroissant, part de variance expliquée
valeurs, vecteurs = ...
part = ...

In [ ]:
def covariance_acp(S_, k):
    """Modèle à k facteurs statistiques : k premières composantes + diagonale résiduelle."""
    val, vec = np.linalg.eigh(S_)
    idx = np.argsort(val)[::-1][:k]
    commun = vec[:, idx] @ np.diag(val[idx]) @ vec[:, idx].T
    return commun + np.diag(np.diag(S_ - commun))

Sigma_acp = covariance_acp(Sigma.values, 3)
print("Écart moyen de corrélation, modèle ACP à 3 facteurs :",
      round(np.abs(en_correlation(Sigma_acp).values - rend.corr().values)[hors_diag].mean(), 3))

### 4.2 Signal ou bruit ? La loi de Marchenko-Pastur

Même sans aucun facteur commun, les valeurs propres d'une matrice des corrélations **estimée** ne valent pas
toutes 1 : elles s'étalent. Pour $N$ titres et $T$ observations indépendantes, elles tombent
approximativement dans l'intervalle de Marchenko et Pastur :

$$\left[(1 - \sqrt{q})^2,\; (1 + \sqrt{q})^2\right], \qquad q = N / T$$

Seules les valeurs propres qui en sortent nettement portent une information (cours, partie 5).

### Exercice 4.2
(a) Calculez l'intervalle pour les dix titres du TD et comptez les valeurs propres de leur matrice des
corrélations qui le dépassent. (b) Simulez 100 titres **sans aucun facteur commun** sur 260 semaines :
les valeurs propres restent-elles dans l'intervalle ? (c) Recommencez avec un facteur de marché et cinq
secteurs : combien de valeurs propres sortent du bruit ?

In [ ]:
def bande_mp(n, t):
    q = n / t
    return ...                               # À COMPLÉTER : bornes basse et haute

# (a) les dix titres du TD
vp = np.sort(np.linalg.eigvalsh(rend.corr().values))[::-1]
bas, haut = bande_mp(N, len(rend))
print(f"(a) Intervalle du bruit pour N = {N}, T = {len(rend)} : [{bas:.2f} ; {haut:.2f}]")
print(f"    Valeurs propres : {np.round(vp, 2)} ; au-dessus du bruit : {(vp > haut).sum()}")

# (b) et (c) : À COMPLÉTER en vous inspirant de la version corrigée

## 5. Le rétrécissement de Ledoit et Wolf (2003)

Plutôt que de choisir entre la matrice empirique $S$, sans biais mais bruitée, et la matrice de Sharpe $F$,
stable mais biaisée, Olivier Ledoit et Michael Wolf les mélangent :

$$\hat\Sigma = \delta\, F + (1 - \delta)\, S$$

L'intensité $\delta$ est estimée à partir des données, pour minimiser l'erreur quadratique attendue.
Dans leur version, le marché est le portefeuille **équipondéré** des titres de l'échantillon.
Le code ci-dessous suit leur estimateur, ligne à ligne.

In [ ]:
def ledoit_wolf_marche(x):
    """Ledoit et Wolf (2003) : rétrécissement de la covariance empirique vers le modèle à un indice.
    x : tableau T x N de rendements. Renvoie la matrice (même unité que x) et l'intensité delta."""
    x = np.asarray(x, dtype=float)
    t, n = x.shape
    x = x - x.mean(axis=0)
    xmkt = x.mean(axis=1)
    echantillon = np.cov(np.column_stack([x, xmkt]), rowvar=False) * (t - 1) / t
    covmkt = echantillon[:n, n]
    varmkt = echantillon[n, n]
    S = echantillon[:n, :n]
    cible = np.outer(covmkt, covmkt) / varmkt
    np.fill_diagonal(cible, np.diag(S))
    c = np.linalg.norm(S - cible, "fro") ** 2
    y = x ** 2
    p = (y.T @ y).sum() / t - (S ** 2).sum()
    rdiag = (y ** 2).sum() / t - (np.diag(S) ** 2).sum()
    z = x * xmkt[:, None]
    v1 = (y.T @ z) / t - covmkt[:, None] * S
    roff1 = (v1 * covmkt[None, :]).sum() / varmkt - (np.diag(v1) * covmkt).sum() / varmkt
    v3 = (z.T @ z) / t - varmkt * S
    roff3 = (v3 * np.outer(covmkt, covmkt)).sum() / varmkt**2 - (np.diag(v3) * covmkt**2).sum() / varmkt**2
    r = rdiag + 2 * roff1 - roff3
    delta = max(0.0, min(1.0, (p - r) / c / t))
    return delta * cible + (1 - delta) * S, delta

Sigma_lw, delta = ledoit_wolf_marche(rend.values)
Sigma_lw *= SEMAINES
print(f"Intensité de rétrécissement estimée : delta = {delta:.2f}")

## 6. Une expérience contrôlée

Sur des données réelles, on ne connaît jamais la vraie matrice de covariance. On simule donc un marché
de **100 titres** dont on connaît la vérité : un facteur de marché, cinq facteurs sectoriels
(20 titres par secteur) et du risque propre. Pour plusieurs longueurs d'historique, on estime la matrice
avec chaque méthode, on construit le portefeuille de variance minimale, et on mesure sa **vraie** volatilité.

Le critère est le rapport entre cette vraie volatilité et la plus faible possible (celle du portefeuille
construit avec la vraie matrice). Un rapport de 1,10 signifie 10 % de risque en trop, dû aux erreurs d'estimation.

Avec 52 semaines pour 100 titres, la matrice empirique n'est même pas inversible : on utilise alors
sa pseudo-inverse, ce qui donne une idée du désastre.

In [ ]:
def gmv(S_):
    x = np.linalg.pinv(S_) @ np.ones(len(S_))
    return x / x.sum()

rng = np.random.default_rng(7)
n_sim, k_sect = 100, 5
secteur_de = np.repeat(np.arange(k_sect), n_sim // k_sect)
b_m = rng.normal(1, 0.25, n_sim)
b_s = rng.uniform(0.6, 1.4, n_sim)
vol_m, vol_s = 0.022, 0.014
vol_e = rng.uniform(0.015, 0.035, n_sim)
B_vrai = np.zeros((n_sim, 1 + k_sect)); B_vrai[:, 0] = b_m
B_vrai[np.arange(n_sim), 1 + secteur_de] = b_s
SF_vrai = np.diag([vol_m**2] + [vol_s**2] * k_sect)
Sigma_vrai = B_vrai @ SF_vrai @ B_vrai.T + np.diag(vol_e**2)
vol_opt = np.sqrt(gmv(Sigma_vrai) @ Sigma_vrai @ gmv(Sigma_vrai))

def estimateurs(x, f):
    S_ = np.cov(x, rowvar=False)
    Xf = np.column_stack([np.ones(len(f)), f])
    c1, *_ = np.linalg.lstsq(Xf[:, :2], x, rcond=None)
    sh = np.var(f[:, 0], ddof=1) * np.outer(c1[1], c1[1]) + np.diag((x - Xf[:, :2] @ c1).var(axis=0, ddof=1))
    c6, *_ = np.linalg.lstsq(Xf, x, rcond=None)
    mf = c6[1:].T @ np.cov(f, rowvar=False) @ c6[1:] + np.diag((x - Xf @ c6).var(axis=0, ddof=1))
    return {"Empirique": S_, "Sharpe (1 facteur)": sh, "6 facteurs": mf,
            "ACP (6 composantes)": covariance_acp(S_, 6), "Ledoit-Wolf": ledoit_wolf_marche(x)[0]}

resultats = {}
for t_obs in (52, 104, 260, 520):
    pertes = {}
    for _ in range(20):
        f = rng.standard_normal((t_obs, 1 + k_sect)) * np.sqrt(np.diag(SF_vrai))
        x = f @ B_vrai.T + rng.standard_normal((t_obs, n_sim)) * vol_e
        for nom, S_ in estimateurs(x, f).items():
            w = gmv(S_)
            pertes.setdefault(nom, []).append(np.sqrt(w @ Sigma_vrai @ w) / vol_opt)
    resultats[f"{t_obs} semaines"] = {k: np.mean(v) for k, v in pertes.items()}
experience = pd.DataFrame(resultats)
experience.style.format("{:.2f}").background_gradient(cmap="Reds", axis=None)

## 7. L'épreuve hors échantillon, sur les vraies données

On répète l'exercice du notebook 2 avec le portefeuille de variance minimale, qui n'utilise que la matrice de covariance :
c'est donc un test pur de la qualité de l'estimation du risque. Fenêtre d'estimation de 2 ans, recalcul toutes les 4 semaines.
Le critère principal est la **volatilité réalisée** : plus elle est faible, meilleure est l'estimation.

In [ ]:
FENETRE, PAS = 104, 4
R, RM, RE = rendements[list(TITRES)], rendements[MARCHE], rendements[list(ETF_SECTEURS)]

def estimer(hist, hm, he):
    x, m = hist.values, hm.values
    S_ = np.cov(x, rowvar=False)
    Xm_ = np.column_stack([np.ones(len(m)), m])
    c1, *_ = np.linalg.lstsq(Xm_, x, rcond=None)
    sh = np.var(m, ddof=1) * np.outer(c1[1], c1[1]) + np.diag((x - Xm_ @ c1).var(axis=0, ddof=1))
    f = np.column_stack([m, residus_sur(he.values, m)])
    Xf = np.column_stack([np.ones(len(m)), f])
    c6, *_ = np.linalg.lstsq(Xf, x, rcond=None)
    mf = c6[1:].T @ np.cov(f, rowvar=False) @ c6[1:] + np.diag((x - Xf @ c6).var(axis=0, ddof=1))
    return {"Empirique": S_, "Sharpe (1 facteur)": sh, "6 facteurs": mf,
            "ACP (3 composantes)": covariance_acp(S_, 3), "Ledoit-Wolf": ledoit_wolf_marche(x)[0]}

rend_oos, dates = {}, []
for k in range(FENETRE, len(R) - 1, PAS):
    S_dict = estimer(R.iloc[k - FENETRE:k], RM.iloc[k - FENETRE:k], RE.iloc[k - FENETRE:k])
    futur = R.iloc[k:k + PAS]
    dates.extend(futur.index)
    for nom, S_ in S_dict.items():
        rend_oos.setdefault(nom, []).extend(futur.values @ gmv(S_))
    rend_oos.setdefault("1/N", []).extend(futur.values.mean(axis=1))
rend_oos = pd.DataFrame(rend_oos, index=dates)
print(f"Période de test : {rend_oos.index[0]:%m/%Y} à {rend_oos.index[-1]:%m/%Y}")
rend_oos.apply(performance).T.sort_values("Volatilité").style.format(
    {"Rendement annuel": "{:.1%}", "Volatilité": "{:.2%}", "Ratio de Sharpe": "{:.2f}", "Perte maximale": "{:.1%}"})

## 8. Bilan

| Modèle | Idée | Force | Faiblesse |
|---|---|---|---|
| Empirique | aucune structure | sans biais | bruité, non inversible si $T < N$ |
| Sharpe (1963) | un seul facteur | très stable, ultra rapide | ignore les secteurs |
| Multi-indices (1967), Barra | facteurs choisis | corrige les secteurs | il faut choisir les facteurs |
| ACP | facteurs extraits des données | aucun choix à faire | facteurs instables, difficiles à interpréter |
| Ledoit-Wolf (2003) | mélange optimal | automatique, robuste | dépend de la cible choisie |

**Pour aller plus loin**

1. Dans l'expérience contrôlée, ajoutez des queues épaisses (loi de Student) aux rendements simulés.
   Le classement change-t-il ?
2. Élargissez l'univers réel à 30 ou 50 titres (voir la liste du notebook 4) et refaites la section 7.
3. Remplacez les facteurs sectoriels par les facteurs de Fama et French, disponibles gratuitement sur le site de Kenneth French.

**Suite** : *Notebook 4 — Le CAPM*, qui transforme le modèle de Sharpe en théorie de l'équilibre des prix.